# Раздел 6 · Sparse-векторы и гибридный поиск

> Этот ноутбук продолжает Раздел 3 («Коллекции и точки») — использует уже созданную там коллекцию `BOOKS_COLLECTION`. Если ниже вылетит `AssertionError: Коллекция ... ещё не создана` — сначала откройте и выполните ноутбук Раздела 3, потом возвращайтесь сюда.

In [1]:
%pip install qdrant-client sentence-transformers ipykernel fastembed --quiet


/home/dimon/vscode/examples_qdrant/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


> Если сразу после установки следующая ячейка падает с `ModuleNotFoundError` на только что установленном пакете — перезапустите ядро (Kernel → Restart) и выполните ячейки заново с начала: иногда ядро не видит пакет, установленный в уже запущенном процессе, без перезапуска.

In [2]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    SparseVectorParams, SparseVector, Prefetch, FusionQuery, Fusion
)

print("OK: библиотеки импортированы")


/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: библиотеки импортированы


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [3]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())


qdrant_lecture
collections=[CollectionDescription(name='exercise1_books'), CollectionDescription(name='exercise1_books_hybrid'), CollectionDescription(name='products'), CollectionDescription(name='products_20260928'), CollectionDescription(name='products_20261001_bm25_fix_ru'), CollectionDescription(name='products_20261002'), CollectionDescription(name='products_base'), CollectionDescription(name='products_cm'), CollectionDescription(name='terraforming_plans')]


In [4]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19546.77it/s]


Размерность эмбеддинга: 384 | книг в подборке: 87


In [5]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


OK: exercise1_books - 87 точек


## Sparse-векторы и гибридный поиск

### Пример — слепое пятно dense-поиска

Произведения `id=64` (поэма Пушкина "Кавказский пленник", 1822) и `id=56` (рассказ Толстого с тем же
названием, 1872) — совершенно разные тексты, но с одинаковым заголовком. Чистый dense-поиск по
названию без уточнения автора:

In [6]:
# запрос состоит буквально из общего названия обоих произведений - специально проверяем "слепое пятно" dense-поиска
blind_spot_hits = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("Кавказский пленник"),
    limit=2,
).points

for h in blind_spot_hits:
    print(h.id, round(h.score, 4), h.payload["author"], "-", h.payload["title"])


64 0.8919 Александр Пушкин - Кавказский пленник
56 0.8878 Лев Толстой - Кавказский пленник


Реальный результат: `id=64` (Пушкин) → `0.8919`, `id=56` (Толстой) → `0.8878` — разрыв всего `0.004`.
Dense честно видит, что оба текста называются одинаково, но почти не может уверенно развести их —
семантика заголовка перевешивает содержание. Обратите внимание: если добавить в запрос автора в
родительном падеже — `"Кавказский пленник Толстого"` — сам по себе dense-поиск это не исправит
(эмбеддинг видит смысл фразы целиком, а не грамматические падежи), а вот sparse/BM25 в таком
падеже тоже "спотыкается" — токен `"Толстого"` не совпадает буквально с `"Толстой"` в тексте.
Работает только точная (именительный падеж) форма — ниже это видно на реальных числах.

### Готовим гибридную коллекцию

Гибридный поиск нужен вектор двух типов сразу — `dense` (как раньше) и `sparse` (BM25). Это готовый
код, просто запустите его — коллекция `HYBRID_COLLECTION` со всеми 87 книгами появится сама,
разбираться в этом не нужно, задание будет чуть ниже:

In [7]:
from fastembed import SparseTextEmbedding

# BM25 как отдельная sparse-модель fastembed - считает классический keyword-скор, не эмбеддинг смысла
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")


def embed_sparse(texts):
    return list(sparse_model.embed(texts))


HYBRID_COLLECTION = PREFIX + "books_hybrid"
if client.collection_exists(HYBRID_COLLECTION):
    client.delete_collection(HYBRID_COLLECTION)

# у коллекции два ИМЕНОВАННЫХ вектора: "dense" (обычный) и "sparse" (через sparse_vectors_config) - это разные типы
client.create_collection(
    collection_name=HYBRID_COLLECTION,
    vectors_config={"dense": VectorParams(size=EMBED_DIM, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams()},
)

# автора добавляем прямо в текст для sparse - иначе BM25 не сможет зацепиться за имя как за отдельный токен
hybrid_texts = [b["title"] + ". " + b["author"] + ". " + b["text"] for b in BOOKS]
hybrid_dense_vectors = embed_passages(hybrid_texts)
hybrid_sparse_vectors = embed_sparse(hybrid_texts)

hybrid_points = []
for b, dvec, svec in zip(BOOKS, hybrid_dense_vectors, hybrid_sparse_vectors):
    hybrid_points.append(PointStruct(
        id=b["id"],
        vector={
            "dense": dvec,
            "sparse": SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()),
        },
        payload={k: v for k, v in b.items() if k != "id"},
    ))

BATCH = 16
for i in range(0, len(hybrid_points), BATCH):
    client.upsert(collection_name=HYBRID_COLLECTION, points=hybrid_points[i:i + BATCH])

print("HYBRID_COLLECTION готова, точек:", client.count(HYBRID_COLLECTION).count)


HYBRID_COLLECTION готова, точек: 87


## Задание 1 — Prefetch + FusionQuery(RRF)

Для запроса `"Кавказский пленник Толстой"` (именно в этой, именительной форме) выполните гибридный поиск
в `HYBRID_COLLECTION`: `Prefetch` по `dense` (`limit=20`) + `Prefetch` по `sparse` (`limit=20`), затем
`query=FusionQuery(fusion=Fusion.RRF)`, `limit=2`. Sparse-вектор запроса посчитайте через
`embed_sparse([query_text])[0]`. Сохраните результат в `hits_ex1`.

**Используйте:** `Prefetch(...)`, `SparseVector(...)`, `FusionQuery(fusion=Fusion.RRF)`, `client.query_points(...)`.

In [10]:
# ВАШ КОД ЗДЕСЬ
query_text = "Кавказский пленник Толстой"   # именительный падеж, одна и та же строка для обоих
dvec = embed_query(query_text)              # внутри припишется "query: "
svec = embed_sparse([query_text])[0]
hits_ex1 = client.query_points(
    collection_name=HYBRID_COLLECTION,
    prefetch=[
        Prefetch(query=dvec, using="dense", limit=20),
        Prefetch(query=SparseVector(indices=svec.indices.tolist(), values=svec.values.tolist()), using="sparse", limit=20),
    ],
     query=FusionQuery(fusion=Fusion.RRF),
    limit=2,
).points

for h in hits_ex1:
    print(h.id, round(h.score, 4), h.payload["author"], "-", h.payload["title"])

56 1.0 Лев Толстой - Кавказский пленник
64 0.6667 Александр Пушкин - Кавказский пленник


In [11]:
def check_ex1(hits_ex1):
    assert len(hits_ex1) == 2
    assert hits_ex1[0].id == 56, (
        "после RRF-гибрида с именем автора в именительном падеже рассказ Толстого (id=56) "
        "должен выйти на первое место, получили top-1 id={}".format(hits_ex1[0].id)
    )
    print("OK: top-1 после гибрида —", hits_ex1[0].payload["title"], "(", hits_ex1[0].payload["author"],
          ") score =", round(hits_ex1[0].score, 3))
    print("ОТВЕТ ДЛЯ STEPIK:", hits_ex1[0].id)


check_ex1(hits_ex1)


OK: top-1 после гибрида — Кавказский пленник ( Лев Толстой ) score = 1.0
ОТВЕТ ДЛЯ STEPIK: 56
